# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from network_connectivity import *

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE or GGminGE
network = "walk"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-1'

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-1'

if territory == 'GGminGE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GGminGE/step-1'
    output_step2_path='../../Data/output/GGminGE/step-2'
    output_step3_path='../../Data/output/GGminGE/step-3'


    save_path = '../../Data/output/GGminGE/step-1'

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading all segments...")
# reLoad segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

Set parameters : GG or GE, bike or walk
Loading all segments...


#### Import des attributs

In [4]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

attributs_info

,Class,meta_attribute,attribute,include_in_index,source_type_GE,source_path_GE,file_name_GE,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size,impact_attribut,file_name,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip,filter_column,filter_values,crs,save_format,Unnamed: 27
1,Commodité,bruit,bruit,True,sitg,attributs/GE/bruit,SPBR_SECTEUR_EXPOSE_AU_BRUIT_2025/SPBR_SECTEUR...,NaN,unavailable,NaN,NaN,0.5,0.5,1.0,defavorable,NaN,polygon,presence,NaN,polygon,presence,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
2,Commodité,temperature,temperature,True,sitg,attributs/GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,opendata,attributs/GG/temperature,20250629_102250.LST.tif,NaN,0.5,0.5,10.0,defavorable,NaN,point,raster,temperature,point,raster,temperature,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
3,Commodité,conflit_md,conflit_md,True,sitg,attributs/GE/network_couche_OCT.shp,RP_final_25112025.shp,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,1.0,defavorable,NaN,line,mean,Partage_us_score,point,sum,severity_score,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
4,Commodité,vegetation,canopee,True,sitg,attributs/GE/canopee,SIPV_ICA_MNC_2023-SHP/SIPV_ICA_MNC_2023.shp,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,10.0,favorable,NaN,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
5,Attractivité,eau,lac_cours_deau,True,sitg,attributs/GE/lac_cours_deau,CAD_NATURE_SOL-SHP/CAD_NATURE_SOL.shp,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50.0,favorable,NaN,polygon,presence,NaN,polygon,presence,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
6,Attractivité,eau,fontaines,True,sitg,unavailable,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50.0,favorable,NaN,point,count,NaN,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
7,Attractivité,espaces_ouverts,espaces_ouverts,True,sitg,attributs/GE/espaces_ouverts,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,NaN,unavailable,NaN,NaN,0.5,0.5,1.0,favorable,NaN,polygon,presence,NaN,polygon,presence,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",prend tous les jardins privés…
8,Attractivité,amenite,rez_actif,True,sitg,attributs/GE/rez_actif,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50.0,favorable,NaN,point,count,NaN,point,count,NaN,10.0,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
9,Attractivité,tp,tp,True,sitg,attributs/GE/tp,TPG_ARRETS-SHP/TPG_ARRETS.shp,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,150.0,favorable,NaN,point,presence,NaN,point,presence,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN
10,Attractivité,amenite,amenite,True,sitg,attributs/GE/amenite,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50.0,favorable,NaN,point,count,NaN,point,count,NaN,10.0,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN


**Connectivité du réseau**

In [5]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

Data initialized
Processing attribute: connectivite
Index(['u', 'v', 'key', 'walk_class', 'source', 'highway', 'osm_id', 'id',
       'element', 'footway', 'sidewalk', 'name', 'maxspeed', 'service',
       'access', 'foot', 'len_m', 'walk_role', 'edge_id', 'length', 'geometry',
       'segment_id'],
      dtype='object')


,u,v,key,walk_class,source,highway,osm_id,id,element,footway,sidewalk,name,maxspeed,service,access,foot,len_m,walk_role,edge_id,length,geometry,segment_id,length_m,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_intersections_in_buffer,conn_branching_in_buffer,conn_index_score,filtered
0,-2673649378734589360,-9140018067926386715,000000,walk_dedicated,OSM_highway_dedicated,footway,way/4272878,4272878.0,way,None,None,None,None,None,None,None,5.920195,dedicated,0,5.920195,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",000000,5.920195,False,True,4,1,0.11125,0.11125,1
1,8023118644210630668,7491611879323938661,000001,walk_dedicated,OSM_highway_dedicated,footway,way/4727161,4727161.0,way,None,None,None,None,None,None,None,759.844982,dedicated,1,8.534589,"LINESTRING (5.99415 46.0667, 5.99426 46.06671)",000001,8.534589,False,True,4,2,0.12250,0.12250,1
2,7491611879323938661,9020566002013775775,000002,walk_dedicated,OSM_highway_dedicated,footway,way/4727161,4727161.0,way,None,None,None,None,None,None,None,759.844982,dedicated,1,50.000000,"LINESTRING (5.99426 46.06671, 5.99431 46.06671...",000002,50.000000,False,True,5,2,0.12250,0.12250,1
3,9020566002013775775,-1711219373344224301,000003,walk_dedicated,OSM_highway_dedicated,footway,way/4727161,4727161.0,way,None,None,None,None,None,None,None,759.844982,dedicated,1,50.000000,"LINESTRING (5.99464 46.06705, 5.99467 46.06721...",000003,50.000000,False,False,2,0,0.10000,0.10000,1
4,-1711219373344224301,-993058096848096498,000004,walk_dedicated,OSM_highway_dedicated,footway,way/4727161,4727161.0,way,None,None,None,None,None,None,None,759.844982,dedicated,1,50.000000,"LINESTRING (5.99462 46.06749, 5.99456 46.06773...",000004,50.000000,False,False,2,0,0.10000,0.10000,1


In [6]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]], attribute)

Geometries cleaned
Filtered data saved for attribute: connectivite in format: parquet
Filtered data saved for attribute: connectivite in format: csv
Filtered data saved for attribute: connectivite in format: gpkg


In [7]:
segmented_net_index.to_crs(target_crs)[["key","segment_id", "geometry","conn_branching_in_buffer","conn_index_score","filtered"]]

,key,segment_id,geometry,conn_branching_in_buffer,conn_index_score,filtered
0,000000,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",0.11125,0.11125,1
1,000001,000001,"LINESTRING (5.99415 46.0667, 5.99426 46.06671)",0.12250,0.12250,1
2,000002,000002,"LINESTRING (5.99426 46.06671, 5.99431 46.06671...",0.12250,0.12250,1
3,000003,000003,"LINESTRING (5.99464 46.06705, 5.99467 46.06721...",0.10000,0.10000,1
4,000004,000004,"LINESTRING (5.99462 46.06749, 5.99456 46.06773...",0.10000,0.10000,1
...,...,...,...,...,...,...
562527,562527,562527,"LINESTRING (6.32438 46.19119, 6.32429 46.19119)",0.17875,0.17875,1
562528,562528,562528,"LINESTRING (6.32429 46.19119, 6.32424 46.19118)",0.17875,0.17875,1
562529,562529,562529,"LINESTRING (6.18578 46.16687, 6.18573 46.16686...",0.15625,0.15625,1
562530,562530,562530,"LINESTRING (6.48129 46.3704, 6.48134 46.37044,...",0.23500,0.23500,1


In [8]:
bike_edges_graph_path = f"{input_file_path}/networkGG/bike_edges_graph.geojson"
bike_edges_graph = gpd.read_file(bike_edges_graph_path)

import json
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

# --- 1) Parse maxspeed -------------------------------------------------------
def parse_maxspeed(v):
    if v is None or pd.isna(v):
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)

    s = str(v).lower().strip()
    if s in {"walk", "walking", "foot"}:
        return 5.0
    if s in {"signals", "none", "variable"}:
        return np.nan

    if "mph" in s:
        m = re.search(r"(\d+)", s)
        return float(m.group(1)) * 1.60934 if m else np.nan

    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan


# --- 2) Highway normalization (list -> str) ----------------------------------
def normalize_highway(v):
    if v is None or pd.isna(v):
        return np.nan
    if isinstance(v, (list, tuple, set)):
        return str(list(v)[0]) if len(v) else np.nan
    return str(v)


# --- 3) Default speeds (CH/urban Europe, adjust if you want) -----------------
DEFAULT_SPEED_BY_HIGHWAY = {
    "motorway": 120, "trunk": 80,
    "primary": 50, "secondary": 50, "tertiary": 50,
    "primary_link": 50, "secondary_link": 50, "tertiary_link": 50,
    "residential": 30, "unclassified": 30,
    "service": 20, "living_street": 20,

    # pedestrian-ish
    "pedestrian": 10,
    "footway": 5, "path": 5, "steps": 5,
    "track": 30,
    "platform": 5, "corridor": 5,
}

ZONE_MAXSPEED_MAP = {
    "CH:urban": 50,
    "CH:rural": 80,
    "CH:motorway": 120,
}


# --- 4) Compute speed_kph on a COPY only ------------------------------------
def compute_speed_kph(df: gpd.GeoDataFrame) -> pd.Series:
    speed = pd.Series(np.nan, index=df.index, dtype="float64")

    # a) use explicit maxspeed tags if present
    for col in ["maxspeed:forward", "maxspeed:backward", "maxspeed"]:
        if col in df.columns:
            speed = speed.fillna(df[col].apply(parse_maxspeed))

    # b) zone-based hints
    if "zone:maxspeed" in df.columns:
        z = df["zone:maxspeed"].astype(str)
        speed = speed.fillna(z.map(ZONE_MAXSPEED_MAP))

    # c) highway fallback
    if "highway" in df.columns:
        hw = df["highway"].apply(normalize_highway)
        speed = speed.fillna(hw.map(DEFAULT_SPEED_BY_HIGHWAY))

    # d) final fallback
    speed = speed.fillna(30.0)

    return speed


# --- 5) Build output gdf and export -----------------------------------------
def export_speed_layer(edges_gdf_graph: gpd.GeoDataFrame, out_dir: Path, basename="edges_speed"):
    out_dir.mkdir(parents=True, exist_ok=True)

    out = edges_gdf_graph.copy()
    out["highway_norm"] = out["highway"].apply(normalize_highway) if "highway" in out.columns else np.nan
    # fill missing highway (optional, conservative)
    out["highway_norm"] = out["highway_norm"].fillna("service")

    out["speed_kph"] = compute_speed_kph(out)

    # keep only useful columns
    keep = [c for c in ["edge_id", "u", "v", "key", "osm_id", "id", "element", "name", "highway", "highway_norm",
                        "maxspeed", "zone:maxspeed", "maxspeed:type", "speed_kph", "length", "len_m", "geometry"]
            if c in out.columns]
    out = out[keep].copy()

    # GeoJSON
    geojson_path = out_dir / f"{basename}.geojson"
    out.to_file(geojson_path, driver="GeoJSON")

    # Parquet (geoparquet)
    parquet_path = out_dir / f"{basename}.parquet"
    out.to_parquet(parquet_path, index=False)

    print("Exported:", geojson_path)
    print("Exported:", parquet_path)
    print("Missing speed_kph:", int(out["speed_kph"].isna().sum()))
    return out



out_dir = Path(f"{input_file_path}/attributs/GG/vitesse")  # choose where you want
speed_gdf = export_speed_layer(bike_edges_graph, out_dir, basename=f"vitesse_all_edges")


Exported: ../../Data/input/attributs/GG/vitesse/vitesse_all_edges.geojson
Exported: ../../Data/input/attributs/GG/vitesse/vitesse_all_edges.parquet
Missing speed_kph: 0
